In [ ]:
import numpy as np
import pandas as pd
import os
import sys
sys.path.append(os.path.join(os.path.abspath(os.getcwd()), "abm_violence"))

In [11]:
def firearms_pipe():
    firearm_licenses = pd.read_csv("data/0124-ffl-list-complete.txt", sep="\t")
    major_licenses = firearm_licenses[firearm_licenses["LIC_TYPE"] == 1 | 2 | 7]
    major_licenses = major_licenses.drop(columns=["LIC_REGN", 
                                                  "LIC_DIST",
                                                  "LIC_XPRDTE", 
                                                  "LIC_SEQN", 
                                                  "PREMISE_STREET", 
                                                  "MAIL_STREET", 
                                                  "MAIL_CITY", 
                                                  "MAIL_STATE", 
                                                  "VOICE_PHONE"])
    major_licenses["PREMISE_CITY"] = major_licenses["PREMISE_CITY"].str.lower()
    major_licenses = major_licenses.groupby(["PREMISE_STATE", "PREMISE_CITY"])[["LICENSE_NAME"]].agg("count")
    return major_licenses.rename(columns={"LICENSE_NAME": "COUNT"}).reset_index()

def how_obtained(s, method_strs): # method_strs is an array
    for method in method_strs:
        if method in s.lower():
            return True
    return False

def mother_jones_pipe():
    # read in file
    mj_shootings = pd.read_csv("data/mother_jones.csv")
    #print("shape: ", mj_shootings.shape)
    # case weapons_obtained_legally as a boolean
    mj_shootings["weapons_obtained_legally"] = mj_shootings["weapons_obtained_legally"].str.lower().replace(
        {"yes": True, "\nyes": True, "no": False, "tbd": "unknown", "-": "unknown"})
    #fix unknown-Unknown issue
    mj_shootings["where_obtained"] = mj_shootings["where_obtained"].replace({"Unknown":"unknown", "Unclear": "unknown"})
    # primitive one-hot-encoding on where_obtained column
    stolen = ["taken", "stolen"]
    mj_shootings["weapon_stolen"] = mj_shootings["where_obtained"
                                                ].apply(lambda x: how_obtained(x, stolen))
    bought = ["purchased", "shop", "pawn", "internet", "show", "store", "retailer", "dealership", "range", "center"]
    mj_shootings["weapon_bought"] = mj_shootings["where_obtained"
                                                          ].apply(lambda x: how_obtained(x, bought))
    # most rows with a specific shop name have a place, place structure
    #Killeen, Guns Galore or city, state
    pat = "([A-Z]+[a-z ]+, [A-Z ]+[ a-zA-Z.]+)"
    shops = mj_shootings["where_obtained"].str.extract(pat).notna()
    mj_shootings["weapon_bought"] = mj_shootings["weapon_bought"] | shops[0]
    family = ["father", "mother", "family"]
    mj_shootings["weapon_family"] = mj_shootings["where_obtained"
                                                          ].apply(lambda x: how_obtained(x, family))
    # separate location
    pat = ", ([A-Za-z\. ]+)"
    states = mj_shootings["location"].str.extract(pat)
    mj_shootings["state"] = states
    pat = "([A-Za-z ]+),"
    cities = mj_shootings["location"].str.extract(pat)
    mj_shootings["city"] = cities    
    # filter out some columns (long text or not relevant)
    mj_shootings = mj_shootings.drop(columns=["location",
                                              "sources", 
                                              "latitude",
                                              "longitude",
                                              "case", # removing case bc the names are brutal
                                              "mental_health_sources", 
                                              "sources_additional_age", 
                                              "mental_health_details", 
                                              "summary",
                                             "race", "gender", "prior_signs_mental_health_issues",
                                             "age_of_shooter", 
                                             "weapon_stolen", "weapon_bought", "weapon_family"])
    # rename columns
    mj_shootings = mj_shootings.rename(columns={"location.1":"location"})
    # canonicalization
    mj_shootings["location"] = mj_shootings["location"].str.lower().str.strip()
    mj_shootings["city"] = mj_shootings["city"].str.lower()
    #display(mj_shootings.tail(10))
    return mj_shootings
    

In [ ]:
def county_populations_pipe():
    county_population = pd.read_csv('data/historical_county_populations_v2.csv')
    pat = "([A-Za-z ]+),"
    counties = county_population["cty"].str.extract(pat)
    pat = ", ([A-Z]+[a-z]+)"
    states = county_population["cty"].str.extract(pat)
    county_population["county"] = counties
    county_population["state"] = states
    return county_population

In [13]:
# display(county_populations_pipe().head())
mj = mother_jones_pipe()
display(mj.head())
firearms = firearms_pipe()
display(firearms.head())

,date,fatalities,injured,total_victims,location,weapons_obtained_legally,where_obtained,weapon_type,weapon_details,type,year,state,city
0,9/4/24,4,9,13,school,unknown,-,semiautomatic rifle,AR-15,mass,2024,Georgia,winder
1,6/21/24,4,10,14,workplace,unknown,-,shotgun; semiautomatic pistol,12-gauge shotgun,mass,2024,Arkansas,fordyce
2,12/6/23,3,1,4,school,unknown,-,semiautomatic handgun,-,mass,2023,Nevada,las vegas
3,10/25/23,18,13,31,other,unknown,Yes,semiautomatic rifle,AR-15-style rifle (Rugar SFAR),Spree,2023,Maine,lewiston
4,8/26/23,3,0,3,workplace,True,local gun stores,"semiautomatic rifle, semiautomatic handgun",AR-15-style rifle; Glock pistol,mass,2023,Florida,jacksonville


/tmp/ipykernel_2841607/3833633899.py:2: DtypeWarning: Columns (11,15) have mixed types. Specify dtype option on import or set low_memory=False.
  firearm_licenses = pd.read_csv("data/0124-ffl-list-complete.txt", sep="\t")


,PREMISE_STATE,PREMISE_CITY,COUNT
0,AK,anchor point,1
1,AK,anchorage,18
2,AK,bethel,2
3,AK,big lake,1
4,AK,chugiak,3


In [55]:
citycountymapper = pd.read_csv("national_place2020.txt", sep='|')[["STATE", "PLACENAME", "COUNTIES"]]
citycountymapper["PLACENAME"] = citycountymapper["PLACENAME"].str.lower()
#citycountymapper["COUNTIES"] = citycountymapper["COUNTIES"].str.lower()
citycountymapper = citycountymapper[citycountymapper["PLACENAME"].str.contains(" city")]
pat = "(.+) city" # there are still some 'city' that are getting through for some reason idk
cities = citycountymapper["PLACENAME"].str.extract(pat)
citycountymapper["PLACENAME"] = cities[0]
citycountymapper = citycountymapper.groupby(["STATE", "PLACENAME"]).agg("first") # some cities span multiple counties.. take the first for simplicity
citycountymapper = citycountymapper.reset_index()
citycountymapper[citycountymapper["COUNTIES"] == "Chambers County"].head()

,STATE,PLACENAME,COUNTIES
244,AL,la fayette,Chambers County
245,AL,lanett,Chambers County
318,AL,valley,Chambers County
8816,TX,anahuac,Chambers County
8856,TX,beach city,Chambers County


In [56]:
#https://www.faa.gov/air_traffic/publications/atpubs/cnt_html/appendix_a.html
# yes i did make this file by hand and i don't want to talk about it 
data=pd.read_csv("state_initials.txt")
stateinitialmapper = pd.DataFrame(data, columns=["state_name", "state_initials"])
stateinitialmapper.head()

,state_name,state_initials
0,Alabama,AL
1,Kentucky,KY
2,Ohio,OH
3,Alaska,AK
4,Louisiana,LA


In [57]:
mj = mj.merge(stateinitialmapper, how="left", left_on="state", right_on="state_name").drop(columns=["state_name"])


mj = mj.merge(citycountymapper, how="left", left_on=["city", "state_initials"], 
              right_on=["PLACENAME", "STATE"]).drop(columns=["STATE"])
mj = mj.merge(firearms, how="left", left_on=["city", "state_initials"], 
              right_on=["PREMISE_CITY", "PREMISE_STATE"])
mj = mj.set_index(["city", "state"])
mj.head()

,,date,fatalities,injured,total_victims,location,weapons_obtained_legally,where_obtained,weapon_type,weapon_details,type,year,state_initials,PLACENAME,COUNTIES,PREMISE_STATE,PREMISE_CITY,COUNT
city,state,,,,,,,,,,,,,,,,,
winder,Georgia,9/4/24,4,9,13,school,unknown,-,semiautomatic rifle,AR-15,mass,2024,GA,winder,Barrow County,GA,winder,3.0
fordyce,Arkansas,6/21/24,4,10,14,workplace,unknown,-,shotgun; semiautomatic pistol,12-gauge shotgun,mass,2024,AR,fordyce,Dallas County,NaN,NaN,NaN
las vegas,Nevada,12/6/23,3,1,4,school,unknown,-,semiautomatic handgun,-,mass,2023,NV,las vegas,Clark County,NV,las vegas,59.0
lewiston,Maine,10/25/23,18,13,31,other,unknown,Yes,semiautomatic rifle,AR-15-style rifle (Rugar SFAR),Spree,2023,ME,lewiston,Androscoggin County,NaN,NaN,NaN
jacksonville,Florida,8/26/23,3,0,3,workplace,True,local gun stores,"semiautomatic rifle, semiautomatic handgun",AR-15-style rifle; Glock pistol,mass,2023,FL,jacksonville,Duval County,FL,jacksonville,43.0


In [14]:
mj.isna().sum()

date                        0
fatalities                  0
injured                     0
total_victims               0
location                    0
weapons_obtained_legally    0
where_obtained              0
weapon_type                 0
weapon_details              1
type                        0
year                        0
state                       0
city                        0
dtype: int64

In [59]:
# manual corrections
firearms[firearms["PREMISE_CITY"]=="honolulu"].loc[1532].to_numpy()

array(['HI', 'honolulu', 8], dtype=object)

In [60]:
citycountymapper[citycountymapper["STATE"]=="HI"]

,STATE,PLACENAME,COUNTIES
2006,HI,lanai,Maui County
2007,HI,pearl,Honolulu County


In [64]:
mj.loc[("trabuco canyon", "California")]["COUNTIES"] = ""

/var/folders/3n/8dzjfpg11pb2g2qd2j1vpwbc0000gn/T/ipykernel_97877/1767628208.py:1: PerformanceWarning: indexing past lexsort depth may impact performance.
  mj.loc[("trabuco canyon", "California")]


,,date,fatalities,injured,total_victims,location,weapons_obtained_legally,where_obtained,weapon_type,weapon_details,type,year,state_initials,PLACENAME,COUNTIES,PREMISE_STATE,PREMISE_CITY,COUNT
city,state,,,,,,,,,,,,,,,,,
trabuco canyon,California,8/23/23,3,6,9,other,unknown,-,semiautomatic handguns; shotgun,-,mass,2023,CA,NaN,NaN,NaN,NaN,NaN


In [61]:
mj[mj["PLACENAME"].isna()][["PLACENAME", "COUNTIES"]]

,,PLACENAME,COUNTIES
city,state,,
trabuco canyon,California,NaN,NaN
nashville,Tennessee,NaN,NaN
hedingham,North Carolina,NaN,NaN
smithsburg,Maryland,NaN,NaN
oxford,Michigan,NaN,NaN
state college,Pennsylvania,NaN,NaN
perryman,Maryland,NaN,NaN
nashville,Tennessee,NaN,NaN
melcroft,Pennsylvania,NaN,NaN
